# Attention Pattern Taxonomy

Classify and catalog attention patterns: diagonal, uniform, sparse,
local, and mixed — with scoring functions for each category.

## Why This Matters

Attention pattern taxonomy reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_taxonomy import (
    pattern_diagonal_score, pattern_uniformity_score,
    pattern_sparsity_score, pattern_locality_score,
    classify_attention_patterns, attention_pattern_taxonomy_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Pattern Classification

Classify each head's pattern into a category.

In [ ]:
for layer in range(4):
    result = classify_attention_patterns(model, tokens, layer=layer)
    print(f"\nLayer {layer}:")
    for h in result['per_head']:
        scores = h['scores']
        print(f"  Head {h['head']}: [{h['category']:8s}] "
              f"diag={scores['diagonal']:.3f} unif={scores['uniform']:.3f} "
              f"sparse={scores['sparse']:.3f} local={scores['local']:.3f}")

## Scoring Functions on Synthetic Patterns

Demonstrate the scoring functions on known pattern types.

In [ ]:
seq = 6
# Diagonal pattern
diag_pat = jnp.zeros((seq, seq))
for i in range(seq):
    if i > 0:
        diag_pat = diag_pat.at[i, i-1].set(1.0)
    else:
        diag_pat = diag_pat.at[0, 0].set(1.0)
print(f"Diagonal score of diagonal pattern: {pattern_diagonal_score(diag_pat):.3f}")
print(f"Uniformity score of diagonal pattern: {pattern_uniformity_score(diag_pat):.3f}")

# Uniform pattern
unif_pat = jnp.zeros((seq, seq))
for i in range(seq):
    unif_pat = unif_pat.at[i, :i+1].set(1.0 / (i + 1))
print(f"\nUniformity score of uniform pattern: {pattern_uniformity_score(unif_pat):.3f}")
print(f"Diagonal score of uniform pattern: {pattern_diagonal_score(unif_pat):.3f}")

## Taxonomy Summary

In [ ]:
result = attention_pattern_taxonomy_summary(model, tokens)
print('Overall distribution:')
for cat, count in sorted(result['overall_distribution'].items()):
    bar = '█' * (count * 3)
    print(f"  {cat:10s}: {count} heads {bar}")
print(f"\nPer-layer:")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: {p['category_counts']}")